In [3]:
# BASE 2 — df_traslados (aristas)
# Objetivo

    # Capturar movimientos reales en la red


# Un traslado es:

# dos episodios consecutivos del mismo paciente en hospitales distintos, temporalmente compatibles

# ✅ Entonces tu máscara debería ser:

# ✔️ obligatorio:

    # hospital cambia
    # existe siguiente episodio
    # ventana temporal razonable (tu ±5 días está bien)

# ✔️ opcional:

    # motivo dice traslado → suma confianza, pero no obligatorio
    # Recomendación fuerte

# esta base responde: “¿cómo fluye la red hospitalaria?”

In [4]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import ast
import pandas as pd
import numpy as np
from src.config import *

In [5]:
df_base = pd.read_excel("../data/final_data/df_base_limpia.xlsx")
df_trayectorias_final = pd.read_excel("../data/final_data/df_trayectorias_final.xlsx")

In [8]:
# 1. 
registros_traslados = []

# Recorremos cada trayectoria limpia
for _, row in df_trayectorias_final.iterrows():
    episodio_id = row['paciente_episodio_id']
    hospitales = row['lista_hospitales']
    
    # Si la lista tiene más de 1 hospital, hubo traslado
    if len(hospitales) > 1:
        # Aislamos los papeles de este episodio, ya ordenados
        historial = df_base_filtrada[df_base_filtrada['paciente_episodio_id'] == episodio_id].sort_values('fecha_ingreso')
        
        for i in range(len(hospitales) - 1):
            h_origen = hospitales[i]
            h_destino = hospitales[i+1]
            
            # Buscamos el último registro en el origen y el primer registro en el destino
            fila_origen = historial[historial['hospital_origen'] == h_origen].iloc[-1]
            fila_destino = historial[historial['hospital_origen'] == h_destino].iloc[0]
            
            registros_traslados.append({
                'paciente_episodio_id': episodio_id,
                'paciente_id': fila_origen['paciente_id'], # Guardamos el original para compatibilidad
                'hospital_origen': h_origen,
                'hospital_destino': h_destino,
                'paso_trayectoria': i + 1,
                'fecha_egreso_origen': fila_origen['fecha_egreso'],
                'fecha_ingreso_destino': fila_destino['fecha_ingreso'],
                'edad_origen': fila_origen['edad'],
                'edad_destino': fila_destino['edad'],
                'motivo_egreso_origen': fila_origen['motivo_egreso']
            })

# Creamos la base cruda de pares
df_pares = pd.DataFrame(registros_traslados)

KeyError: 'lista_hospitales'

In [6]:
# 2. APLICACIÓN DE TU LÓGICA DE TIEMPOS Y FLAGS (MÚSCULO ANALÍTICO)
# ==============================================================================
TOL_DELTA_SEGUNDOS = 60 # Tu tolerancia
KEYWORDS_TRASLADO = ['traslado', 'derivacion', 'derivación', 'referencia', 'transferencia']

# A. Normalización y Flag de Motivo
df_pares['motivo_egreso_norm'] = df_pares['motivo_egreso_origen'].astype(str).str.lower().str.strip()
df_pares['flag_motivo_traslado'] = df_pares['motivo_egreso_norm'].apply(
    lambda x: 1 if any(k in x for k in KEYWORDS_TRASLADO) else 0
)

# B. Cálculo de Deltas Temporales
df_pares['delta_segundos'] = (df_pares['fecha_ingreso_destino'] - df_pares['fecha_egreso_origen']).dt.total_seconds()

# Corrección por tolerancia
df_pares['delta_segundos_corr'] = df_pares['delta_segundos'].where(
    df_pares['delta_segundos'].abs() > TOL_DELTA_SEGUNDOS, 0
)

# Deltas en Días y Horas
df_pares['delta_dias'] = (df_pares['delta_segundos_corr'] / 86400).round().astype('Int64')
df_pares['delta_horas'] = df_pares['delta_segundos_corr'] / 3600

# Delta de Edad
df_pares['delta_edad'] = df_pares['edad_destino'] - df_pares['edad_origen']

# C. Clasificación de Traslado (Tu lógica de negocio)
df_pares['tipo_traslado'] = np.select(
    [
        df_pares['delta_segundos_corr'] == 0,
        (df_pares['delta_horas'] > 0) & (df_pares['delta_horas'] <= 24),
        df_pares['delta_horas'] > 24
    ],
    [
        'inmediato',   # <= tolerancia
        'rapido',      # dentro de 24h
        'diferido'     # más de 24h
    ],
    default='otro'     # Negativos o nan
)

# D. Creación de Flags de Calidad
df_pares['flag_fechas_faltantes'] = df_pares['fecha_egreso_origen'].isna() | df_pares['fecha_ingreso_destino'].isna()
df_pares['flag_overlap'] = df_pares['delta_dias'] < 0
df_pares['flag_overlap_extremo'] = df_pares['delta_dias'] < -2
df_pares['flag_gap_corto'] = df_pares['delta_dias'].between(0, 2)
df_pares['flag_gap_medio'] = df_pares['delta_dias'].between(3, 5)
df_pares['flag_gap_largo'] = df_pares['delta_dias'] > 5
df_pares['flag_edad_inconsistente'] = df_pares['delta_edad'].abs() > 2
df_pares['flag_inmediato'] = df_pares['tipo_traslado'] == 'inmediato'
df_pares['flag_diferido'] = df_pares['tipo_traslado'] == 'diferido'


NameError: name 'df_pares' is not defined

In [ ]:
# 3. FILTRADO FINAL (LOOSE VS STRICT)
# ==============================================================================
# Filtros base: No fechas faltantes, ventana -5 a 30 días, edades consistentes
cond_base = (
    (~df_pares['flag_fechas_faltantes']) &
    (df_pares['delta_dias'] >= -5) &
    (df_pares['delta_dias'] <= 30) &
    (~df_pares['flag_edad_inconsistente'])
)

# Base LOOSE (Solo cumple continuidad lógica y temporal, sin importar qué tipeó el administrativo)
df_traslados_loose = df_pares[cond_base].copy()

# Base STRICT (Cumple lo anterior Y además el administrativo anotó la palabra clave)
cond_motivo = df_traslados_loose['flag_motivo_traslado'] == 1
df_traslados_strict = df_traslados_loose[cond_motivo].copy()

# Columnas Finales ordenadas
cols_finales = [
    'paciente_episodio_id', 'paciente_id', 'hospital_origen', 'hospital_destino', 'paso_trayectoria',
    'fecha_egreso_origen', 'fecha_ingreso_destino', 'delta_dias', 'delta_horas', 'tipo_traslado',
    'edad_origen', 'edad_destino', 'delta_edad', 'motivo_egreso_origen', 'flag_motivo_traslado',
    'flag_overlap', 'flag_overlap_extremo', 'flag_gap_corto', 'flag_gap_medio', 'flag_gap_largo',
    'flag_inmediato', 'flag_diferido'
]

df_traslados_loose = df_traslados_loose[cols_finales]
df_traslados_strict = df_traslados_strict[cols_finales]

# Exportar
df_traslados_loose.to_excel("../data/final_data/df_traslados_loose.xlsx", index=False)
df_traslados_strict.to_excel("../data/final_data/df_traslados_strict.xlsx", index=False)

print(f"Traslados extraídos de trayectorias (Loose): {len(df_traslados_loose)}")
print(f"Traslados validados por palabra clave (Strict): {len(df_traslados_strict)}")

In [ ]:
registros_traslados = []

# Recorremos cada trayectoria limpia
for _, row in df_trayectorias_limpias.iterrows():
    episodio_id = row['paciente_episodio_id']
    hospitales = row['lista_hospitales']
    
    # Si la lista tiene más de 1 hospital, hubo traslado
    if len(hospitales) > 1:
        # Aislamos los papeles de este episodio, ya ordenados
        historial = df_base_filtrada[df_base_filtrada['paciente_episodio_id'] == episodio_id].sort_values('fecha_ingreso')
        
        for i in range(len(hospitales) - 1):
            h_origen = hospitales[i]
            h_destino = hospitales[i+1]
            
            # Buscamos el último registro en el origen y el primer registro en el destino
            fila_origen = historial[historial['hospital_origen'] == h_origen].iloc[-1]
            fila_destino = historial[historial['hospital_origen'] == h_destino].iloc[0]
            
            registros_traslados.append({
                'paciente_episodio_id': episodio_id,
                'paciente_id': fila_origen['paciente_id'], # Guardamos el original para compatibilidad
                'hospital_origen': h_origen,
                'hospital_destino': h_destino,
                'paso_trayectoria': i + 1,
                'fecha_egreso_origen': fila_origen['fecha_egreso'],
                'fecha_ingreso_destino': fila_destino['fecha_ingreso'],
                'edad_origen': fila_origen['edad'],
                'edad_destino': fila_destino['edad'],
                'motivo_egreso_origen': fila_origen['motivo_egreso']
            })

# Creamos la base cruda de pares
df_pares = pd.DataFrame(registros_traslados)

In [ ]:
# 2. NORMALIZACION DE MOTIVO DE EGRESO
# ==============================================================================

def normalizar_motivo(x):
    if pd.isna(x):
        return np.nan
    x = str(x).lower().strip()
    return x

df_base['motivo_egreso_norm'] = df_base['motivo_egreso'].apply(normalizar_motivo)

# palabras clave de traslado (ajustar según dataset real)
KEYWORDS_TRASLADO = [
    'traslado',
    'derivacion',
    'derivación',
    'referencia',
    'transferencia'
]

def es_traslado(motivo):
    if pd.isna(motivo):
        return 0
    return int(any(k in motivo for k in KEYWORDS_TRASLADO))

df_base['flag_motivo_traslado'] = df_base['motivo_egreso_norm'].apply(es_traslado)

In [ ]:
# # 3. DEDUPLICACION TECNICA INGRESO Y EGRESO IGUAL
# # ==============================================================================

# cols_dup = [
#     'paciente_id',
#     'hospital_origen',
#     'fecha_ingreso',
#     'fecha_egreso'
# ]

# df_base['flag_duplicado_exacto'] = df_base.duplicated(subset=cols_dup, keep='first')

# # para construir traslados usamos una version sin duplicados exactos
# df_nodup = df_base[~df_base['flag_duplicado_exacto']].copy()


# ==============================================================================
# 3. DEDUPLICACION POR CERCANIA TEMPORAL (INGRESO Y EGRESO CERCA)
# ==============================================================================

# ordenar bien
df_base = df_base.sort_values([
    'paciente_id',
    'hospital_origen',
    'fecha_ingreso',
    'fecha_egreso'
])

# shift dentro de paciente + hospital
df_base['prev_ingreso'] = df_base.groupby(
    ['paciente_id', 'hospital_origen']
)['fecha_ingreso'].shift(1)

df_base['prev_egreso'] = df_base.groupby(
    ['paciente_id', 'hospital_origen']
)['fecha_egreso'].shift(1)

# diferencias en segundos
df_base['diff_ingreso_seg'] = (
    df_base['fecha_ingreso'] - df_base['prev_ingreso']
).dt.total_seconds().abs()

df_base['diff_egreso_seg'] = (
    df_base['fecha_egreso'] - df_base['prev_egreso']
).dt.total_seconds().abs()

TOL_SEGUNDOS = 60  # podés probar 30, 60, 120
# FLAG DUPLICADO
df_base['flag_duplicado_cercano'] = (
    (df_base['diff_ingreso_seg'] <= TOL_SEGUNDOS) &
    (df_base['diff_egreso_seg'] <= TOL_SEGUNDOS)
)
# CONSTRUCCION SIN ESTOS DUPLICADOS
df_nodup = df_base[~df_base['flag_duplicado_cercano']].copy()

In [ ]:
# 4. CONSTRUCCION DE PARES CONSECUTIVOS
# ==============================================================================

df_nodup = df_nodup.sort_values(['paciente_id', 'fecha_ingreso'])

# shift
df_nodup['hospital_destino'] = df_nodup.groupby('paciente_id')['hospital_origen'].shift(-1)
df_nodup['fecha_ingreso_destino'] = df_nodup.groupby('paciente_id')['fecha_ingreso'].shift(-1)
df_nodup['edad_destino'] = df_nodup.groupby('paciente_id')['edad'].shift(-1)

# también motivo del origen ya lo tenemos

In [ ]:
# 5. VARIABLES DERIVADAS (DELTA ROBUSTO)
# ==============================================================================

# delta en segundos (base)
df_nodup['delta_segundos'] = (
    df_nodup['fecha_ingreso_destino'] - df_nodup['fecha_egreso']
).dt.total_seconds()

# corrección por tolerancia
df_nodup['delta_segundos_corr'] = df_nodup['delta_segundos'].where(
    df_nodup['delta_segundos'].abs() > TOL_DELTA_SEGUNDOS,
    0
)

# delta en días ENTEROS (evita floats raros)
df_nodup['delta_dias'] = (
    df_nodup['delta_segundos_corr'] / (60 * 60 * 24)
).round().astype('Int64')

# delta edad
df_nodup['delta_edad'] = df_nodup['edad_destino'] - df_nodup['edad']


# ==============================================================================
# 5.B CLASIFICACION TEMPORAL DEL TRASLADO
# ==============================================================================

# en horas (más interpretable que segundos)
df_nodup['delta_horas'] = df_nodup['delta_segundos_corr'] / 3600

# clasificacion
df_nodup['tipo_traslado'] = np.select(
    [
        df_nodup['delta_segundos_corr'] == 0,
        (df_nodup['delta_horas'] > 0) & (df_nodup['delta_horas'] <= 24),
        df_nodup['delta_horas'] > 24
    ],
    [
        'inmediato',   # mismo evento (<= 5 min)
        'rapido',      # dentro del mismo día
        'diferido'     # más de 1 día
    ],
    default='otro'  # incluye negativos raros o edge cases
)

In [ ]:
#  6. FLAGS DE CALIDAD
# ==============================================================================

df_nodup['flag_mismo_hospital'] = (
    df_nodup['hospital_origen'] == df_nodup['hospital_destino']
)

df_nodup['flag_fechas_faltantes'] = (
    df_nodup['fecha_egreso'].isna() |
    df_nodup['fecha_ingreso_destino'].isna()
)

df_nodup['flag_overlap'] = df_nodup['delta_dias'] < 0
df_nodup['flag_overlap_extremo'] = df_nodup['delta_dias'] < -2

df_nodup['flag_gap_corto'] = df_nodup['delta_dias'].between(0, 2)
df_nodup['flag_gap_medio'] = df_nodup['delta_dias'].between(3, 5)
df_nodup['flag_gap_largo'] = df_nodup['delta_dias'] > 5

df_nodup['flag_edad_inconsistente'] = df_nodup['delta_edad'].abs() > 2

df_nodup['flag_inmediato'] = df_nodup['tipo_traslado'] == 'inmediato'
df_nodup['flag_diferido'] = df_nodup['tipo_traslado'] == 'diferido'

In [ ]:
#  7. FILTROS BASE (SIN MOTIVO)
# ==============================================================================

cond_base = (
    (df_nodup['hospital_destino'].notna()) &
    (~df_nodup['flag_mismo_hospital']) &
    (~df_nodup['flag_fechas_faltantes']) &
    (df_nodup['delta_dias'] >= -5) &
    (df_nodup['delta_dias'] <= 30) &
    (~df_nodup['flag_edad_inconsistente'])
)

df_pares_validos = df_nodup[cond_base].copy()

In [ ]:
# 8. DF TRASLADOS LOOSE
# ==============================================================================

df_traslados_loose = df_pares_validos.copy()

In [ ]:
# 9. DF TRASLADOS STRICT
# ==============================================================================

cond_motivo = df_pares_validos['flag_motivo_traslado'] == 1

df_traslados_strict = df_pares_validos[cond_motivo].copy()

In [ ]:
# 10. COLUMNAS FINALES
# ==============================================================================

cols_finales = [
    'paciente_id',
    'hospital_origen',
    'hospital_destino',
    'fecha_egreso',
    'fecha_ingreso_destino',
    'delta_dias',
    'edad',
    'edad_destino',
    'delta_edad',
    'flag_motivo_traslado',
    'flag_overlap',
    'flag_overlap_extremo',
    'flag_gap_corto',
    'flag_gap_medio',
    'flag_gap_largo',
    'flag_edad_inconsistente',
    'delta_segundos_corr',
    'tipo_traslado',
    'delta_horas',
    'flag_inmediato',
    'flag_diferido'
]

df_traslados_loose = df_traslados_loose[cols_finales].rename(columns={
    'fecha_egreso': 'fecha_egreso_origen'
})

df_traslados_strict = df_traslados_strict[cols_finales].rename(columns={
    'fecha_egreso': 'fecha_egreso_origen'
})

In [ ]:
df_traslados_loose.to_excel("../data/final_data/df_traslados_loose.xlsx", index=False)

df_traslados_strict.to_excel("../data/final_data/df_traslados_strict.xlsx", index=False)

# df_nodup.to_parquet("../data/debug/df_nodup_debug.parquet")

### verificaciones y explroacion

In [ ]:
# SANITY CHECKS GENERALES
# ==============================================================================

print("=== TAMAÑOS ===")
print("Base original:", len(df_base))
print("Sin duplicados:", len(df_nodup))
print("Traslados loose:", len(df_traslados_loose))
print("Traslados strict:", len(df_traslados_strict))


print("\n=== DELTA DIAS (LOOSE) ===")
print(df_traslados_loose['delta_dias'].describe())

print("\n=== DELTA DIAS (STRICT) ===")
print(df_traslados_strict['delta_dias'].describe())


print("\n=== DELTA EDAD ===")
print(df_traslados_loose['delta_edad'].describe())


print("\n=== PROPORCION CON MOTIVO ===")
print(df_traslados_loose['flag_motivo_traslado'].value_counts(normalize=True))


print("\n=== EJEMPLOS RANDOM ===")
display(df_traslados_loose.sample(5))


print("\n=== CHEQUEO CLAVE: hospitales distintos ===")
print((df_traslados_loose['hospital_origen'] == df_traslados_loose['hospital_destino']).sum())


print("\n=== CHEQUEO EDAD ===")
print((df_traslados_loose['delta_edad'].abs() > 2).sum())


print("\n=== OVERLAPS ===")
print(df_traslados_loose['flag_overlap'].value_counts())


print("\n=== TOP FLUJOS A->B ===")
print(
    df_traslados_loose
    .groupby(['hospital_origen', 'hospital_destino'])
    .size()
    .sort_values(ascending=False)
    .head(10)
)

In [ ]:
print("\n=== DISTRIBUCION POR DELTA ===")
print(df_traslados_loose['delta_dias'].value_counts().sort_index())

In [ ]:
print("\n=== TIPOS DE TRASLADO ===")
print(df_traslados_loose['tipo_traslado'].value_counts(normalize=True))